In [1]:
import pandas as pd

In [3]:
clusters = pd.read_csv('../data/annotation/euk_GH18/domain_analyses/euk_clusters.csv')
cid = pd.read_csv('../data/annotation/euk_GH18/domain_analyses/euk_CID_hmmsearch.csv')
cid = cid.sort_values(by=['seq_score', 'target_name'], ascending=[False, True]).reset_index(drop=True)
cid = cid.drop_duplicates(subset='target_name', keep='first')

annot = pd.read_csv('../data/annotation/euk_GH18/domain_analyses/all_domains_annotated_deduplicated_cleaned_proteinfprintGH18_taxonomy_promandHK.csv')

cid_clusters = clusters.merge(cid[['target_name','query_name']], left_on='fastaID',right_on='target_name',how='left')
annot_cid_clust = cid_clusters.merge(annot, right_on='protein_accession',left_on='Uniprot_Accessions')
annot_cid_clust = annot_cid_clust.drop_duplicates(subset='fastaID')
annot_cid_clust

/var/folders/fg/5tjhqq3n0t70943jzd8k47800000gp/T/ipykernel_29694/749289398.py:6: DtypeWarning: Columns (28: subspecies, 32: superphylum) have mixed types. Specify dtype option on import or set low_memory=False.
  annot = pd.read_csv('../data/annotation/euk_GH18/domain_analyses/all_domains_annotated_deduplicated_cleaned_proteinfprintGH18_taxonomy_promandHK.csv')


,Unnamed: 0_x,id,61.0,111.0,188.0,285.0,361.0,fastaID,Uniprot_Accessions,target_name,...,superclass,superfamily,superorder,superphylum,clade_ordered,no rank_ordered,non_GH18_fingerprint,GH18_fingerprint,promiscuous,housekeeper
0,0,0,0,0.0,3.0,7.0,5.0,A0A010Q896|1of1,A0A010Q896,A0A010Q896|1of1,...,NaN,NaN,NaN,NaN,Opisthokonta; saccharomyceta; leotiomyceta; so...,cellular organisms; Colletotrichum acutatum sp...,NaN,GH18,NaN,NaN
1,1,1,0,0.0,3.0,7.0,5.0,A0A010QB65|1of1,A0A010QB65,A0A010QB65|1of1,...,NaN,NaN,NaN,NaN,Opisthokonta; saccharomyceta; leotiomyceta; so...,cellular organisms; Colletotrichum acutatum sp...,PF00187,GH18 PF00187,NaN,T
2,2,10,0,0.0,0.0,22.0,104.0,A0A016S9Z0|1of1,A0A016S9Z0,A0A016S9Z0|1of1,...,NaN,Strongyloidea,NaN,NaN,Opisthokonta; Eumetazoa; Bilateria; Protostomi...,cellular organisms,NaN,GH18,NaN,NaN
3,3,100,0,1.0,1.0,5.0,8.0,A0A060SC99|1of1,A0A060SC99,A0A060SC99|1of1,...,NaN,NaN,NaN,NaN,Opisthokonta,cellular organisms; Agaricomycetes incertae sedis,NaN,GH18,NaN,NaN
4,4,1000,0,0.0,0.0,6.0,7.0,A0A0D2N077|1of1,A0A0D2N077,A0A0D2N077|1of1,...,NaN,NaN,NaN,NaN,Opisthokonta,cellular organisms,NaN,GH18,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29880,29880,9995,7,1993.0,657.0,4016.0,2072.0,A0A835W011|2of4,A0A835W011,NaN,...,NaN,NaN,NaN,NaN,core chlorophytes; CS clade,cellular organisms,SSF57016 SSF57016 SSF57016 SSF57016,GH18 GH18 GH18 GH18 SSF57016 SSF57016 SSF57016...,NaN,T
29881,29881,9996,7,382.0,1817.0,6212.0,5254.0,A0A835W011|3of4,A0A835W011,NaN,...,NaN,NaN,NaN,NaN,core chlorophytes; CS clade,cellular organisms,SSF57016 SSF57016 SSF57016 SSF57016,GH18 GH18 GH18 GH18 SSF57016 SSF57016 SSF57016...,NaN,T
29882,29882,9997,7,1994.0,658.0,4017.0,2073.0,A0A835W011|4of4,A0A835W011,NaN,...,NaN,NaN,NaN,NaN,core chlorophytes; CS clade,cellular organisms,SSF57016 SSF57016 SSF57016 SSF57016,GH18 GH18 GH18 GH18 SSF57016 SSF57016 SSF57016...,NaN,T
29883,29883,9998,0,0.0,22.0,41.0,79.0,A0A835W823|1of1,A0A835W823,A0A835W823|1of1,...,NaN,NaN,NaN,NaN,core chlorophytes; CS clade,cellular organisms,PS50287 PF00530 SSF69318 PF00059 SSF56436 G3DS...,GH18 PS50287 PF00530 SSF69318 PF00059 SSF56436...,T,NaN


In [4]:
annot_cid_clust['Uniprot_Accessions'].nunique()

27437

summary

In [20]:
import pandas as pd

CLUSTER_COL = '111.0'
top_clusters = [0, 1, 2, 3,4,5,6,7,8,9,10]

df = annot_cid_clust

sub = df[df[CLUSTER_COL].isin(top_clusters)].copy()

# Flag logic: counting any non-null value as "flagged".
# If these columns use a specific marker, replace with e.g.:
#   prom_mask = sub['promiscuous'].eq('T')
prom_mask = sub['promiscuous'].notna()
hk_mask   = sub['housekeeper'].notna()

summary = (
    sub.assign(promiscuous_flag=prom_mask, housekeeper_flag=hk_mask)
       .groupby([CLUSTER_COL, 'query_name'])
       .agg(n_sequences=('query_name', 'size'),
            n_promiscuous=('promiscuous_flag', 'sum'),
            n_housekeeper=('housekeeper_flag', 'sum'))
       .reset_index()
       .sort_values([CLUSTER_COL, 'n_sequences'], ascending=[True, False])
)
summaryking = (
    sub.assign(promiscuous_flag=prom_mask, housekeeper_flag=hk_mask)
       .groupby([CLUSTER_COL, 'kingdom'])
       .agg(n_sequences=('kingdom', 'size'),
            n_promiscuous=('promiscuous_flag', 'sum'),
            n_housekeeper=('housekeeper_flag', 'sum'))
       .reset_index()
       .sort_values([CLUSTER_COL, 'n_sequences'], ascending=[True, False])
)

summaryphy = (
    sub.assign(promiscuous_flag=prom_mask, housekeeper_flag=hk_mask)
       .groupby([CLUSTER_COL, 'phylum'])
       .agg(n_sequences=('phylum', 'size'),
            n_promiscuous=('promiscuous_flag', 'sum'),
            n_housekeeper=('housekeeper_flag', 'sum'))
       .reset_index()
       .sort_values([CLUSTER_COL, 'n_sequences'], ascending=[True, False])
)

# Per-cluster totals across all query_names
cluster_totals = (
    sub.assign(promiscuous_flag=prom_mask, housekeeper_flag=hk_mask)
       .groupby(CLUSTER_COL)
       .agg(total_sequences=('query_name', 'size'),
            total_promiscuous=('promiscuous_flag', 'sum'),
            total_housekeeper=('housekeeper_flag', 'sum'))
       .reset_index()
)

print("Per query_name within each cluster:")
display(summary)
print('kingdom breakdown')
display(summaryking)
print('phylum breakdown')
display(summaryphy)
print("\nCluster totals:")
display(cluster_totals)

Per query_name within each cluster:


,111.0,query_name,n_sequences,n_promiscuous,n_housekeeper
1,0.0,CID3,18086,351,7335
2,0.0,CID4,350,16,89
4,0.0,CID6,63,0,9
5,0.0,CID7,57,0,5
0,0.0,CID1and0,51,0,0
3,0.0,CID5,7,0,0
6,1.0,CID1and0,4551,75,909
7,1.0,CID2,1,0,0
8,1.0,CID4,1,0,0
11,2.0,CID7,1010,2,188


kingdom breakdown


,111.0,kingdom,n_sequences,n_promiscuous,n_housekeeper
0,0.0,Fungi,8693,87,3082
1,0.0,Metazoa,7830,114,4241
2,0.0,Viridiplantae,1765,138,53
3,1.0,Fungi,2535,37,816
5,1.0,Viridiplantae,2046,38,71
4,1.0,Metazoa,52,0,23
7,2.0,Metazoa,640,0,187
8,2.0,Viridiplantae,240,1,2
6,2.0,Fungi,80,1,0
9,3.0,Viridiplantae,734,0,434


phylum breakdown


,111.0,phylum,n_sequences,n_promiscuous,n_housekeeper
3,0.0,Ascomycota,6005,77,2536
2,0.0,Arthropoda,4222,55,2328
5,0.0,Basidiomycota,1810,6,392
33,0.0,Streptophyta,1695,107,26
10,0.0,Chordata,1629,27,1185
...,...,...,...,...,...
110,6.0,Nematoda,1,0,0
116,7.0,Oomycota,79,0,1
117,8.0,Arthropoda,79,0,0
118,9.0,Nematoda,62,1,2



Cluster totals:


,111.0,total_sequences,total_promiscuous,total_housekeeper
0,0.0,18624,367,7438
1,1.0,4639,75,912
2,2.0,1021,2,190
3,3.0,734,0,434
4,4.0,724,26,86
5,5.0,533,3,52
6,6.0,451,2,11
7,7.0,79,0,1
8,8.0,79,0,0
9,9.0,62,1,2


In [21]:
import pandas as pd

CLUSTER_COL = '188.0'
top_clusters = [0, 1, 2, 3,4,5,6,7,8,9,10]

df = annot_cid_clust

sub = df[df[CLUSTER_COL].isin(top_clusters)].copy()

# Flag logic: counting any non-null value as "flagged".
# If these columns use a specific marker, replace with e.g.:
#   prom_mask = sub['promiscuous'].eq('T')
prom_mask = sub['promiscuous'].notna()
hk_mask   = sub['housekeeper'].notna()

summary = (
    sub.assign(promiscuous_flag=prom_mask, housekeeper_flag=hk_mask)
       .groupby([CLUSTER_COL, 'query_name'])
       .agg(n_sequences=('query_name', 'size'),
            n_promiscuous=('promiscuous_flag', 'sum'),
            n_housekeeper=('housekeeper_flag', 'sum'))
       .reset_index()
       .sort_values([CLUSTER_COL, 'n_sequences'], ascending=[True, False])
)
summaryking = (
    sub.assign(promiscuous_flag=prom_mask, housekeeper_flag=hk_mask)
       .groupby([CLUSTER_COL, 'kingdom'])
       .agg(n_sequences=('kingdom', 'size'),
            n_promiscuous=('promiscuous_flag', 'sum'),
            n_housekeeper=('housekeeper_flag', 'sum'))
       .reset_index()
       .sort_values([CLUSTER_COL, 'n_sequences'], ascending=[True, False])
)

summaryphy = (
    sub.assign(promiscuous_flag=prom_mask, housekeeper_flag=hk_mask)
       .groupby([CLUSTER_COL, 'phylum'])
       .agg(n_sequences=('phylum', 'size'),
            n_promiscuous=('promiscuous_flag', 'sum'),
            n_housekeeper=('housekeeper_flag', 'sum'))
       .reset_index()
       .sort_values([CLUSTER_COL, 'n_sequences'], ascending=[True, False])
)

# Per-cluster totals across all query_names
cluster_totals = (
    sub.assign(promiscuous_flag=prom_mask, housekeeper_flag=hk_mask)
       .groupby(CLUSTER_COL)
       .agg(total_sequences=('query_name', 'size'),
            total_promiscuous=('promiscuous_flag', 'sum'),
            total_housekeeper=('housekeeper_flag', 'sum'))
       .reset_index()
)

print("Per query_name within each cluster:")
display(summary)
print('kingdom breakdown')
display(summaryking)
print('phylum breakdown')
display(summaryphy)
print("\nCluster totals:")
display(cluster_totals)

Per query_name within each cluster:


,188.0,query_name,n_sequences,n_promiscuous,n_housekeeper
1,0.0,CID3,15879,301,6841
2,0.0,CID4,255,14,79
4,0.0,CID6,7,0,5
5,0.0,CID7,5,0,2
3,0.0,CID5,4,0,0
0,0.0,CID1and0,3,0,0
6,1.0,CID1and0,4230,74,888
7,1.0,CID2,1,0,0
8,1.0,CID4,1,0,0
10,2.0,CID7,897,0,186


kingdom breakdown


,188.0,kingdom,n_sequences,n_promiscuous,n_housekeeper
0,0.0,Fungi,7536,78,2800
1,0.0,Metazoa,6820,102,4032
2,0.0,Viridiplantae,1554,109,41
3,1.0,Fungi,2371,36,799
5,1.0,Viridiplantae,1855,38,68
4,1.0,Metazoa,36,0,22
7,2.0,Metazoa,606,0,185
8,2.0,Viridiplantae,223,0,1
6,2.0,Fungi,39,0,0
9,3.0,Fungi,721,8,248


phylum breakdown


,188.0,phylum,n_sequences,n_promiscuous,n_housekeeper
2,0.0,Ascomycota,5097,68,2258
1,0.0,Arthropoda,3704,47,2170
4,0.0,Basidiomycota,1714,6,388
28,0.0,Streptophyta,1534,101,26
9,0.0,Chordata,1507,26,1171
...,...,...,...,...,...
87,7.0,Hemichordata,1,0,0
89,7.0,Nematoda,1,0,0
95,8.0,Streptophyta,115,0,12
96,9.0,Arthropoda,101,1,96



Cluster totals:


,188.0,total_sequences,total_promiscuous,total_housekeeper
0,0.0,16153,315,6927
1,1.0,4268,74,891
2,2.0,901,0,187
3,3.0,721,8,248
4,4.0,705,26,86
5,5.0,513,3,51
6,6.0,470,0,402
7,7.0,394,2,9
8,8.0,115,0,12
9,9.0,101,1,96


promiscuity calc and plotting

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

# 1) Parameters
input_file = "../data/annotation/euk_GH18/domain_analyses/converted_protein_domains_interproscan_GH18replacedfprint.tsv"  # or your GH18‐filtered file
gh18_id     = "GH18"
output_file = "../data/annotation/euk_GH18/domain_analyses/260716partner_stats_converted_protein_domains_interproscan_GH18replacedfprint.tsv"

# 2) Load and filter
df = pd.read_csv(input_file, sep="\t",
                 
                 dtype=str)
df["domains"] = df["domains_interproscan"].str.split()
gh = df[df["domains"].apply(lambda L: gh18_id in L)].copy()
N = len(gh)

# 3) Aggregate stats
n_GH18 = defaultdict(int)
neighbor_sets = defaultdict(set)

for domains in gh["domains"]:
    # unique partners per protein
    ups = {d for d in domains if d != gh18_id}
    for p in ups:
        n_GH18[p] += 1
    # record neighbors within each GH18 protein
    for i, dom in enumerate(domains):
        if dom == gh18_id:
            continue
        left  = domains[i-1] if i>0 else None
        right = domains[i+1] if i<len(domains)-1 else None
        for neigh in (left, right):
            if neigh and neigh != dom:
                neighbor_sets[dom].add(neigh)

# 4) Compute T and normalized sums
T_GH18 = {d: len(v) for d, v in neighbor_sets.items()}
sum_T = sum(T_GH18.values())

# 5) Build results
records = []
for dom in sorted(n_GH18):
    n_i = n_GH18[dom]
    T_i = T_GH18.get(dom, 0)
    pi_i = T_i / sum_T if sum_T>0 else 0.0
    f_i  = n_i / N
    phi_i = pi_i * np.log(pi_i / f_i) if pi_i>0 and f_i>0 else 0.0
    records.append({
        "Domain":   dom,
        "n_GH18":   n_i,
        "T_GH18":   T_i,
        "pi_GH18":  pi_i,
        "phi_GH18": phi_i
    })

partner_stats = pd.DataFrame(records, columns=["Domain","n_GH18","T_GH18","pi_GH18","phi_GH18"])

# 6) Save and preview
partner_stats.to_csv(output_file, sep="\t", index=False)
partner_stats.head()
